# Fixed nested ToolRet candidate pools

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [ ]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'AGENTS.md').exists(), 'Set DIFFJEV_ROOT to the checked-out repository'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev import workflows as w
from diffjev.config import load_yaml_config
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']


In [ ]:
# Reuse the frozen manifests from notebook 01; no negative mining on test outcomes.
for backbone in cfg['backbones']:
    methods = ['diffjev','serialized_dlm'] + (['ar'] if backbone.startswith('nemotron') else [])
    for method in methods:
        runner = w.load_runner(backbone,method=method,seed=SEED,device=DEVICE)
        w.evaluate_scaling(runner,w.run_directory(backbone,method,seed=SEED)/'evaluation')
        del runner
        gc.collect()

In [ ]:
for name in cfg['references'] + ['bge']:
    runner=w.baseline_runner(name,device=DEVICE)
    w.evaluate_scaling(runner,ROOT/'outputs/formal/references'/name)
    del runner
    gc.collect()

K > 32 is cardinality extrapolation. Reports include actual K, Hit@1, coverage, raw query predictions, and stagewise gold survival. These are supplied-candidate routing results, not full-corpus retrieval or tool execution.